In [1]:
import json, math
from pathlib import Path
root=Path('.')
dataset=json.loads((root/'evidence/evaluation_40.json').read_text(encoding='utf-8'))
baseline=json.loads((root/'evidence/v0_results.json').read_text(encoding='utf-8'))
improved=json.loads((root/'evidence/v4_results.json').read_text(encoding='utf-8'))
assert len(dataset['cases'])==40 and len(baseline)==len(improved)==120
assert {(r['case_id'],r['repeat_index']) for r in baseline}=={(r['case_id'],r['repeat_index']) for r in improved}
print('Verified: 40 identical cases, three repeats per version.')

Verified: 40 identical cases, three repeats per version.


In [2]:
def measure(rows):
    fraud=[r for r in rows if r['expected']['fraud_flag']]
    latency=sorted(r['latency_ms'] for r in rows)
    return {'intent':sum(r['checks']['routing_exact'] for r in rows)/len(rows),'journey':sum(r['passed'] for r in rows)/len(rows),'fraud_recall':sum(r['actual']['route']['potential_fraud_error'] for r in fraud)/len(fraud),'p95_ms':latency[math.ceil(.95*len(rows))-1],'provider_USD':sum(r['actual']['model_cost_usd'] for r in rows)}
a,b=measure(baseline),measure(improved)
print(json.dumps({'baseline':a,'improved':b,'delta':{k:b[k]-a[k] for k in a}},indent=2))

{
  "baseline": {
    "intent": 0.85,
    "journey": 0.8,
    "fraud_recall": 0.8,
    "p95_ms": 30.95,
    "provider_USD": 0.0
  },
  "improved": {
    "intent": 1.0,
    "journey": 0.975,
    "fraud_recall": 1.0,
    "p95_ms": 30.577,
    "provider_USD": 0.0
  },
  "delta": {
    "intent": 0.15000000000000002,
    "journey": 0.17499999999999993,
    "fraud_recall": 0.19999999999999996,
    "p95_ms": -0.37299999999999756,
    "provider_USD": 0.0
  }
}


In [3]:
old={(r['case_id'],r['repeat_index']):r for r in baseline}
regressions=sorted({r['case_id'] for r in improved if old[(r['case_id'],r['repeat_index'])]['passed'] and not r['passed']})
remaining=sorted({r['case_id'] for r in improved if not r['passed']})
assert regressions==[] and remaining==['RG-056']
print(json.dumps({'regressions':regressions,'remaining_failures':remaining}))

{"regressions": [], "remaining_failures": ["RG-056"]}


In [4]:
print('These are mock-rule measurements, not GLM improvement evidence.')
print('Human labels: pending. LLM judge/human agreement: not scored.')
print('Provider cost excludes CPU, infrastructure and human review. Loom: not recorded.')

These are mock-rule measurements, not GLM improvement evidence.
Human labels: pending. LLM judge/human agreement: not scored.
Provider cost excludes CPU, infrastructure and human review. Loom: not recorded.
